# Валидация данных для ML

В этом ноутбуке произведу валидацию данных для подготовки датасета для алгоритма "Случайный лес". На данный момент в new_sales_data.csv находится 17754 строк. Это информация по продажам за 2023-2025 года. Информация помесячная, т.е. 12 записей за год на один артикул.

## Задачи данного блока
Фильтрация:
- фильтрация "плохих" товаров (которые встречаются редко или у них плохая история)
- xyz анализ для понимания стабильности

Обучение ML:
- выбрать 5-10 товаров, которые будем прогнозировать (скорее всего 3X и 3Y, но можно попробовать и парочку Z, чтобы понять на сколько модель ошибается)
- выбрать признаки по которым будем обучать модель. 
    - Предположительно для MVP 1: 
        - lag_1, lag_2, lag_3, lag_6, lag_12
        - ma3, ma6 для того чтобы модель видела тренд
        - month_sin = sin(2π * month / 12) для определения текущего месяца и спроса в этот месяц
- в файлах лежит срин с возможными признаками. Оттуда можно брать их и экспериментировать

In [36]:
import pandas as pd
from pathlib import Path

data_path = Path.cwd().parent.parent / "data" / "clean" / "new_sales_data.csv"

df_sales = pd.read_csv(data_path)

df_sales.head()

,product,sku,qty,unit,month
0,КОНФ ВЕС Столичные,КО01828,63.586,кг,2023-12
1,КОНФ ВЕС Сибирский Сувенир,НС07823,42.809,кг,2023-12
2,КОНФ ВЕС Тамбовский волк Люкс,ТК07914,44.556,кг,2023-12
3,КОНФ ВЕС Сибирский сувенир Кедровый грильяж,НС20906,21.408,кг,2023-12
4,КОНФ ВЕС БУТЫЛОЧКИ С КОНЬЯКОМ,ЯП24671,22.819,кг,2023-12


In [37]:
# CV по каждому товару (sku) по месяцам
cv_by_sku = (
    df_sales.groupby('sku')['qty']
    .agg(mean_qty='mean', std_qty='std', count='count')
    .assign(cv=lambda x: x['std_qty'] / x['mean_qty'])
    .reset_index()
)
# если нужно добавить cv обратно в df_sales
df_sales = df_sales.merge(cv_by_sku[['sku', 'cv', 'count']], on='sku', how='left')
df_sales


,product,sku,qty,unit,month,cv,count
0,КОНФ ВЕС Столичные,КО01828,63.586,кг,2023-12,0.362193,35.0
1,КОНФ ВЕС Сибирский Сувенир,НС07823,42.809,кг,2023-12,0.614627,35.0
2,КОНФ ВЕС Тамбовский волк Люкс,ТК07914,44.556,кг,2023-12,0.480439,35.0
3,КОНФ ВЕС Сибирский сувенир Кедровый грильяж,НС20906,21.408,кг,2023-12,0.468089,35.0
4,КОНФ ВЕС БУТЫЛОЧКИ С КОНЬЯКОМ,ЯП24671,22.819,кг,2023-12,0.651500,26.0
...,...,...,...,...,...,...,...
17749,"Мороженое КОРОВКА ИЗ КОРЕНОВКИ ""Пломбир"" шокол...",NaN,7.000,шт,2025-10,NaN,NaN
17750,Эскимо Коровка из Кореновки пломбир МОЛ ШОК Бе...,NaN,4.000,шт,2025-10,NaN,NaN
17751,Рожок КОРОВКА ИЗ КОРЕНОВКИ пломбир с брусничны...,NaN,2.000,шт,2025-10,NaN,NaN
17752,"Мороженое КОРОВКА ИЗ КОРЕНОВКИ ""Эскимо"" в шоко...",NaN,1.000,шт,2025-10,NaN,NaN


In [38]:
def make_classifier(x_thresh: float, y_thresh: float, min_months: int = 3):
    """Фабрика функций-классификаторов XYZ.

    Возвращает функцию, которую можно применять к строке DataFrame (row),
    она использует пороги x_thresh и y_thresh и минимальное количество месяцев.
    """
    def classify(row):
        cv = row["cv"]
        count = row["count"]

        # Проверяем количество месяцев
        if count < min_months:
            return "Unknown"

        # Проверяем CV
        if pd.isna(cv):
            return "Unknown"
        elif cv <= x_thresh:
            return "X - Стабильный"
        elif cv <= y_thresh:
            return "Y - Средний"
        else:
            return "Z - Нестабильный"

    return classify

stat_clf = make_classifier(x_thresh=0.3, y_thresh=0.6, min_months=3)
xyz_analysis = df_sales.apply(stat_clf, axis=1)
df_sales['xyz_analysis'] = xyz_analysis
df_sales.head()

,product,sku,qty,unit,month,cv,count,xyz_analysis
0,КОНФ ВЕС Столичные,КО01828,63.586,кг,2023-12,0.362193,35.0,Y - Средний
1,КОНФ ВЕС Сибирский Сувенир,НС07823,42.809,кг,2023-12,0.614627,35.0,Z - Нестабильный
2,КОНФ ВЕС Тамбовский волк Люкс,ТК07914,44.556,кг,2023-12,0.480439,35.0,Y - Средний
3,КОНФ ВЕС Сибирский сувенир Кедровый грильяж,НС20906,21.408,кг,2023-12,0.468089,35.0,Y - Средний
4,КОНФ ВЕС БУТЫЛОЧКИ С КОНЬЯКОМ,ЯП24671,22.819,кг,2023-12,0.651500,26.0,Z - Нестабильный


In [39]:
df_sales[df_sales['sku'] == 'КО01828']


,product,sku,qty,unit,month,cv,count,xyz_analysis
0,КОНФ ВЕС Столичные,КО01828,63.586,кг,2023-12,0.362193,35.0,Y - Средний
550,КОНФ ВЕС Столичные,КО01828,61.472,кг,2023-06,0.362193,35.0,Y - Средний
985,КОНФ ВЕС Столичные,КО01828,41.806,кг,2023-08,0.362193,35.0,Y - Средний
1412,КОНФ ВЕС Столичные,КО01828,41.056,кг,2023-05,0.362193,35.0,Y - Средний
1858,КОНФ ВЕС Столичные,КО01828,53.896,кг,2023-02,0.362193,35.0,Y - Средний
2264,КОНФ ВЕС Столичные,КО01828,59.392,кг,2023-04,0.362193,35.0,Y - Средний
2722,КОНФ ВЕС Столичные,КО01828,68.412,кг,2023-10,0.362193,35.0,Y - Средний
3224,КОНФ ВЕС Столичные,КО01828,46.110,кг,2023-03,0.362193,35.0,Y - Средний
3658,КОНФ ВЕС Столичные,КО01828,36.074,кг,2023-01,0.362193,35.0,Y - Средний
4070,КОНФ ВЕС Столичные,КО01828,43.060,кг,2023-11,0.362193,35.0,Y - Средний
